In [2]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 

from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
from tqdm.auto import tqdm
import librosa

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


# Make stimuli for 2024 mono unfamiliar language distractor experiment

## As of jun 21 - re make using screened set 
___
## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-talker
___
Will be using precut manifest of stimuli for cues, foregrounds, and distractors from manifest:
`/om/user/imgriff/datasets/human_distractor_language_2024/final_safe_stim_manifest_w_cue_tg_lang_dists_w_transcripts.pdpkl`

Manifest generated in notebook `gen_different_language_distractor_expmt_stim_2024.ipynb`

English distractors from SWC stimuli (same used in other SWC experiments)

Cut distractors using Chinese mandarin and dutch:
Source from common voice:
- zh-CN is Chinese mandarin subset `/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/zh-CN` 
- nl is Dutch subset `/om2/data/public/mozilla-CommonVoice-9.0/cv-corpus-9.0-2022-04-27/nl` 

Will be using dev split from each language experiment stimuli
___
Stim will be moved to `/om/user/imgriff/datasets/human_distractor_language_2024`
___

### Def audio read fn

## Load and prep distractor manifests

#### Get max n targets from set of foregrounds

In [10]:
stim_out_path = Path('/om/user/imgriff/datasets/human_distractor_language_2024')
## Load model stim df 

## Write out good_model_df 
fname = stim_out_path / 'final_safe_stim_manifest_w_cue_tg_lang_dists_w_transcripts.pdpkl'
manifest = pd.read_pickle(fname)


In [2]:
old_manifests = pd.read_pickle('/om/user/imgriff/datasets/human_distractor_language_2024/human_expmt_manifest_w_transcripts.pdpkl')


In [3]:
old_manifests

,target_sr,experiment_key_target_word_ix,cue_sr,target_fn,cue_fn,word,word_int,condition,snr,src_ix,...,target_transcripts,distractor_transcripts,zh_distractor_client_id,zh_distractor_gender,zh_distractor_src_fn,nl_distractor_client_id,nl_distractor_gender,nl_distractor_src_fn,nl_distractor_corpus,zh_distractor_corpus
0,44100,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,above,1,1-talker,0,703492,...,"[extend, from, above, the, navel, to, below]","[after, details, were, known, by]",357beaf8df6e188414fab6927ff4c2d213227a22c463f8...,male,/om/user/imgriff/datasets/human_distractor_lan...,f8ad661e8869636d8ad664e7d6eef0ac256ad42f53937c...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
1,44100,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,according,3,1-talker,0,244898,...,"[according, to, environment]","[as, on, board]",53879004792fa1d04e34c220163d4a8aa827059df7ffac...,male,/om/user/imgriff/datasets/human_distractor_lan...,8f7b7a60719ade866b8973c73fe1a90ac6d6cb6a0de519...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
2,44100,2,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,across,4,1-talker,0,126322,...,"[its, visible, across, interstellar]","[while, waiting, for, hooker, to, return]",4383308152f190d56fd4b614c494cd4c2b0f966d9a9089...,male,/om/user/imgriff/datasets/human_distractor_lan...,3108ab73b3f900eacdc24c1d12c7f765b3fe0705a6c624...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
3,44100,3,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,action,5,1-talker,0,937425,...,"[so, urgent, was, the, action, that, the, wick...","[the, military, outposts, on, the]",902ec60c0f7f0bfc60543185758f96d7097021c837ef14...,male,/om/user/imgriff/datasets/human_distractor_lan...,e4c1db01d08617e90d96f377ab8195bd01c25816bfd78d...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
4,44100,4,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,activities,7,1-talker,0,940458,...,"[enjoyed, activities, as, well]","[whom, he, had, also, worked, as, a, drafter]",4a4e4a24326b8112dc1770c72a4e7334e5debd0096daee...,male,/om/user/imgriff/datasets/human_distractor_lan...,fd2b26246cb5d2d044f044ba73488940e14aeb0c1c756a...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,44100,355,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,world,787,1-talker,0,378537,...,"[in, reshaping, the, world, consequences, for]","[ethical, political, or, technological]",498078b833c1d4e6e4495d995305b5ae436c11fa68e721...,female,/om/user/imgriff/datasets/human_distractor_lan...,149737bec6714c7e83e87c4113a1eabdd127a4c3fd0b93...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
356,44100,356,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,writing,791,1-talker,0,708274,...,"[humanism, writing, in, nineteen]","[lutheran, church]",24e6cf94b008beb2a07429524f1e449b4c6c4e169f872c...,female,/om/user/imgriff/datasets/human_distractor_lan...,ebdba6c88c07dee4f58f8248c3a9ed948aa9438233e5d8...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
357,44100,357,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,written,792,1-talker,0,714687,...,"[of, letters, which, endure, in, the, babingto...","[of, migratory, birds, and, game]",8a5b0f4a00967db460276340db49bcf2bec8d41b91c8cc...,female,/om/user/imgriff/datasets/human_distractor_lan...,52112f4c966d9e2246d170b0d2b6ee6886999b9827df6e...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
358,44100,358,44100,/om/user/

#### Make sure distractor sex still matches distribution

In [7]:
dist_sex_manifest = manifest[['distractor_gender','nl_distractor_gender', 'zh_distractor_gender']]
dist_sex_manifest.eq(dist_sex_manifest.iloc[:, 0], axis=0).all(1).all()

True

In [15]:
## get n unique words per distractor gender 

manifest.groupby(['distractor_gender', 'nl_distractor_gender', 'zh_distractor_gender']).word.nunique()

distractor_gender  nl_distractor_gender  zh_distractor_gender
female             female                female                  360
male               male                  male                    360
Name: word, dtype: int64

In [20]:
### Add same/different sex condition to manifest per language 

manifest['en_sex_cond'] = manifest.gender == manifest.distractor_gender
manifest['zh_sex_cond'] = manifest.gender == manifest.zh_distractor_gender
manifest['nl_sex_cond'] = manifest.gender == manifest.nl_distractor_gender

# for each sex cond, make True = same and False = different
manifest['en_sex_cond'] = manifest['en_sex_cond'].map({True: 'same', False: 'different'})
manifest['zh_sex_cond'] = manifest['zh_sex_cond'].map({True: 'same', False: 'different'})
manifest['nl_sex_cond'] = manifest['nl_sex_cond'].map({True: 'same', False: 'different'})



In [34]:
all(manifest['en_sex_cond'] == manifest['zh_sex_cond']), all(manifest['en_sex_cond'] == manifest['nl_sex_cond'])

(True, True)

### Cut samples for human experiment make gender balanced and 50/50 same different sex

In [87]:
np.random.seed(6) ## seed 6 gives random sampling that achieves balance of same/different x male/female 

n_words = manifest.word.nunique()
n_per_cut = n_words// 4 # split into 4 cuts

# Iteratively sample to achieve desired balance  
## sample male examples 
used_words = []

male_manifest = manifest[manifest['gender'] == 'male']
male_same_df = male_manifest[(male_manifest['en_sex_cond'] == 'same') & (~male_manifest.word.isin(used_words))].sample(n_per_cut, replace=False)
used_words.extend(male_same_df.word.unique())  

male_diff_df = male_manifest[(male_manifest['en_sex_cond'] == 'different') & (~male_manifest.word.isin(used_words))].sample(n_per_cut, replace=False)
used_words.extend(male_diff_df.word.unique())  

male_df = pd.concat([male_diff_df, male_same_df], axis=0, ignore_index=True)

female_manifest = manifest[manifest['gender'] == 'female']
# unique_words = manifest.word.unique()
# sample unique words 
female_same_df = female_manifest[(female_manifest['en_sex_cond'] == 'same') & (~female_manifest.word.isin(used_words))].sample(n_per_cut, replace=False)
used_words.extend(female_same_df.word.unique())
female_diff_df = female_manifest[(female_manifest['en_sex_cond'] == 'different') & (~female_manifest.word.isin(used_words))].sample(n_per_cut, replace=False)
used_words.extend(female_diff_df.word.unique())

female_df = pd.concat([female_diff_df, female_same_df], axis=0, ignore_index=True)

sampled_manifest = pd.concat([male_df, female_df], axis=0, ignore_index=True)

# make sure words are unique 
sampled_manifest.word.nunique()

360

In [88]:
all(sampled_manifest['en_sex_cond'] == sampled_manifest['zh_sex_cond']), all(sampled_manifest['en_sex_cond'] == sampled_manifest['nl_sex_cond'])

(True, True)

In [95]:
sampled_manifest.zh_distractor_src_fn[0]

PosixPath('/om/user/imgriff/datasets/human_distractor_language_2024/zh_distractor_excerpts/common_voice_zh-CN_22365575_3_sec.wav')

#### Sort so word order matches key

In [50]:
## Sort sampled_manifest by word 
sampled_manifest.sort_values(by='word', inplace=True)
sampled_manifest.reset_index(drop=True, inplace=True)
word_list = sampled_manifest.word

In [ ]:
## check if we need to resave the word list - only needed if new words or new order
with open(stim_out_path /  "human_distractor_language_word_key.pkl", 'rb') as f:
    word_key = pickle.load(f)

for i, word in enumerate(word_key.values()):
    if not word == word_list[i]:
        print(i, word)

In [55]:
## Save updated human sampled manifest 
# sampled_manifest.to_pickle(stim_out_path / "human_expmt_manifest_w_transcripts_safe_examples.pdpkl" )

In [23]:
stim_out_path / "human_expmt_manifest_w_transcripts_safe_examples.pdpkl"

PosixPath('/om/user/imgriff/datasets/human_distractor_language_2024/human_expmt_manifest_w_transcripts_safe_examples.pdpkl')

In [24]:
sampled_manifest = pd.read_pickle(stim_out_path / "human_expmt_manifest_w_transcripts_safe_examples.pdpkl" )

### Prep condition list

In [11]:
# Make condition dict 
import itertools 
import pickle 
import numpy as np 
snrs = np.arange(-9, 4, 3).tolist() # -9, -6, -3, 0, 3

conditions = ['english_1-distractor',
              'dutch_1-distractor',
              'mandarin_1-distractor']

condition_pairs = list(itertools.product(conditions, snrs))
condition_pairs.append(('SILENCE', 'inf'))
cond_ix_dict = {ix:cond for ix, cond in enumerate(condition_pairs)}

out_name = stim_out_path / "human_distractor_language_cond_map.pkl"

# already exists, written in previous notebook 
 
# write condition dict as pickle 
# with open(out_name, 'wb') as f:
#     pickle.dump(cond_ix_dict, f)


In [38]:
## Make word Key .js from pickle of word dict
import IPython.display as ipd


In [41]:
ipd.Audio("/om/user/imgriff/datasets/human_distractor_language_2024/sounds/condition_00/105.wav")

In [40]:
human_exp_manifest.loc[105, 'word']

'english'

## Make sure stimuli generation worked 

In [65]:
paths = list(Path('/om/user/imgriff/datasets/human_distractor_language_2024/sounds/').rglob('*/*.wav'))

In [63]:
import librosa

In [69]:
bad_paths = []
for path in paths:
    try:
        librosa.get_duration(filename=path.as_posix())
    except Exception as e:
        print(e)
        bad_paths.append(path)
print(len(bad_paths))

0


## Screen for bad talkers and make full model df 

In [90]:
df = pd.read_pickle(stim_out_path / "human_expmt_manifest_w_transcripts.pdpkl" )

In [13]:
bad_talker_names = ['wikipedia', 'viktor-o-ledenyov',  'acerperi', 'bowie-media', '1904-cc' ]

In [17]:
ix = 131

eg = df.loc[ix]

ipd.display(ipd.Audio(eg.cue_fn))
ipd.display(ipd.Audio(eg.target_fn))

In [19]:
df[df.client_id.isin(bad_talker_names)]

,target_sr,experiment_key_target_word_ix,cue_sr,target_fn,cue_fn,word,word_int,condition,snr,src_ix,...,target_transcripts,distractor_transcripts,zh_distractor_client_id,zh_distractor_gender,zh_distractor_src_fn,nl_distractor_client_id,nl_distractor_gender,nl_distractor_src_fn,nl_distractor_corpus,zh_distractor_corpus
98,44100,98,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,doctor,195,1-talker,0,367752,...,"[its, in, the, medical, doctor, and, doctor]","[to, architect, thomas, jefferson]",d18792b4453d6ed867570462cd26b6c9937288e9fd9319...,male,/om/user/imgriff/datasets/human_distractor_lan...,892a82d6ec7fd2f56c26450278edbe013f77066182ba4b...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
131,44100,131,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,games,279,1-talker,0,536737,...,"[during, home, games, at, the, staples]","[examples, of, the, types, of, animal, cow]",2100a627b7e17b575fd25619681d5c75116e7cdd119991...,male,/om/user/imgriff/datasets/human_distractor_lan...,cdd10f681f8a0c4d0d9a74010cc24a5135eb6ef7efd43a...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
149,44100,149,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,immediately,324,1-talker,0,536957,...,"[and, immediately, began, landing]","[institutions, like, international, monetary, ...",c3f2acbffa9fad6600f6ea707a195058cc3c1c524be35d...,male,/om/user/imgriff/datasets/human_distractor_lan...,892a82d6ec7fd2f56c26450278edbe013f77066182ba4b...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
193,44100,193,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,music,440,1-talker,0,998941,...,"[hushscaled, vocal, music, creation]","[wealth, could, reward, hard, work]",844bed009831973f43495eab1ca52ab865d6350c406b38...,female,/om/user/imgriff/datasets/human_distractor_lan...,23571776619d54ea7ce5bc6f27c20e742f650187824dae...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
251,44100,251,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,published,547,1-talker,0,376903,...,"[and, published, in, one thousand, nine hundre...","[extensively, by, television, and, newspapers]",844bed009831973f43495eab1ca52ab865d6350c406b38...,female,/om/user/imgriff/datasets/human_distractor_lan...,57c2b9cdc436aef96a2df07996e042e87b660ea3c20c71...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
276,44100,276,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,science,598,1-talker,0,370278,...,"[organizing, science, with]","[polipods, created, by, hasbro]",ab5ab1f7d75838fc0e4d14c931d66de698bd60426f8089...,female,/om/user/imgriff/datasets/human_distractor_lan...,52112f4c966d9e2246d170b0d2b6ee6886999b9827df6e...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
284,44100,284,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,simply,623,1-talker,0,993029,...,"[so, that, they, can, simply, load, in, the, c...","[of, the, core, rule, books, were, slightly, r...",04776675c17170276c1037565d816677753f4bb875be70...,female,/om/user/imgriff/datasets/human_distractor_lan...,493a95c46abb885f230c8ec86e26ab3403148b6410ae89...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv
309,44100,309,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,/om/user/imgriff/datasets/spatial_audio_pipeli...,success,676,1-talker,0,378655,...,"[well, be, at, corporate, sas, as, you, mentio...","[two, words, are, sometimes, hyphenated]",7974762cb54aa5347f075c9be16cd7bd8fc39e9e4c53b8...,female,/om/user/imgriff/datasets/human_distractor_lan...,48ade8319f3820b7cb61c89b2eeb286ade347a0fb9438b...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv


In [11]:
## Look at model df 

model_df = pd.read_pickle(stim_out_path / 'final_stim_manifest_w_cue_tg_lang_dists.pdpkl')

### Add columns from screened stimuli

In [15]:
## Load screened full manifest 
# /om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl
out_name = "/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl"
full_manifest = pd.read_pickle(out_name)


In [16]:
full_manifest.head()

,orig_full_index,distractor_client_id,distractor_clip_dur_in_s,distractor_clip_end_in_s,distractor_clip_start_in_s,distractor_corpus,distractor_gender,distractor_gender_int,distractor_split,distractor_split_int,...,cue_clip_end_in_s,gender_cond_td,cue_clip_dur_in_s,target_transcripts,distractor_transcripts,word_int,dist_word_int,target_f0,cue_f0,distractor_f0
0,0,popularoutcast,0.44,359.92,359.48,swc,female,0,NaN,0,...,1147.96,female_female,0.45,"[however, the, cost, for, these, new, types, o...","[made, from, either, dna, or]",703,210,181.518509,175.263489,189.355301
1,1,matthewdgonzalez,0.49,584.65,584.16,swc,male,1,NaN,0,...,1147.96,female_male,0.45,"[however, the, cost, for, these, new, types, o...","[in, one thousand, nine hundred and forty, sev...",703,610,181.518509,175.263489,130.063400
2,2,flyingtoaster,0.17,1711.42,1711.25,swc,female,0,NaN,0,...,1906.30,female_female,0.49,"[so, that, they, can, simply, load, in, the, c...","[of, the, core, rule, books, were, slightly, r...",623,764,185.744934,168.147736,196.366379
3,3,warmvoiceover,0.76,487.07,486.31,swc,male,1,NaN,0,...,1906.30,female_male,0.49,"[so, that, they, can, simply, load, in, the, c...","[philadelphia, created, in, seventeen]",623,166,185.744934,168.147736,107.241776
4,4,popularoutcast,0.56,432.21,431.65,swc,female,0,NaN,0,...,643.09,male_female,0.17,"[resulted, in, the, death, due, to, a, gas, leak]","[this, was, a, then, unknown, fault]",175,747,90.175484,87.435898,222.476532


In [17]:
## Remove rows from model df that are missing from the screened manifest - can use orig_full_index column in full manifest 

# just get example still contained - these should be fine, but will check here 
possible_bad_eg = model_df[(model_df.index.isin(full_manifest.orig_full_index)) & (model_df.client_id.isin(bad_talker_names))]

possible_bad_eg ## These are good - screened by listeneing 

,distractor_client_id,distractor_corpus,distractor_gender,distractor_gender_int,distractor_sr,distractor_src_fn,distractor_word,src_ix,client_id,corpus,...,zh_distractor_gender,zh_distractor_src_fn,nl_distractor_client_id,nl_distractor_gender,nl_distractor_src_fn,nl_distractor_corpus,zh_distractor_corpus,signal_len_s,word_int,distractor_word_int
0,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,either,992601,1904-cc,swc,...,female,/om/user/imgriff/datasets/human_distractor_lan...,45cd595afd01e386500dfeff20804f2b8cc3d0fc0382cb...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,703,210
1,matthewdgonzalez,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,seven,992601,1904-cc,swc,...,male,/om/user/imgriff/datasets/human_distractor_lan...,3108ab73b3f900eacdc24c1d12c7f765b3fe0705a6c624...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,703,610
2,flyingtoaster,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,were,993029,1904-cc,swc,...,female,/om/user/imgriff/datasets/human_distractor_lan...,493a95c46abb885f230c8ec86e26ab3403148b6410ae89...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,623,764
3,warmvoiceover,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,created,993029,1904-cc,swc,...,male,/om/user/imgriff/datasets/human_distractor_lan...,55edb0faa36b1882a71261e93de75db781f61128b3876c...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,623,166
116,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,white,536957,bowie-media,swc,...,female,/om/user/imgriff/datasets/human_distractor_lan...,ac56a26bf01bd0bbbfeb5396f37d21b7e63bb94d31c011...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,324,772
117,viktor-o-ledenyov,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,international,536957,bowie-media,swc,...,male,/om/user/imgriff/datasets/human_distractor_lan...,892a82d6ec7fd2f56c26450278edbe013f77066182ba4b...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,324,341
120,laurahale,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,united,537232,bowie-media,swc,...,female,/om/user/imgriff/datasets/human_distractor_lan...,149737bec6714c7e83e87c4113a1eabdd127a4c3fd0b93...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,249,745
121,karltalk,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,james,537232,bowie-media,swc,...,male,/om/user/imgriff/datasets/human_distractor_lan...,cdd10f681f8a0c4d0d9a74010cc24a5135eb6ef7efd43a...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,249,351
122,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,about,536542,bowie-media,swc,...,female,/om/user/imgriff/datasets/human_distractor_lan...,23571776619d54ea7ce5bc6f27c20e742f650187824dae...,female,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,452,0
123,enviroboy,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,continue,536542,bowie-media,swc,...,male,/om/user/imgriff/datasets/human_distractor_lan...,1cae50ccac9c88a8596023c782950eb5bb9a1d95be6148...,male,/om/user/imgriff/datasets/human_distractor_lan...,cv,cv,2,452,151


In [92]:
##  Listened to all of them - they're good
# 
ix = 15
eg = possible_bad_eg.iloc[ix]
ipd.display(ipd.Audio(eg.cue_src_fn))
ipd.display(ipd.Audio(eg.src_fn))


In [20]:
## Cut and add transcripts 

good_model_df = model_df.loc[model_df.index.isin(full_manifest.orig_full_index)].copy()
good_model_df
# get target and english distractor transcripts from orig_full_index
good_model_df.loc[: , 'target_transcripts'] = full_manifest.loc[full_manifest.orig_full_index.isin(good_model_df.index), 'target_transcripts'].values
good_model_df.loc[: , 'distractor_transcripts'] = full_manifest.loc[full_manifest.orig_full_index.isin(good_model_df.index), 'distractor_transcripts'].values
good_model_df.loc[: , 'target_f0'] = full_manifest.loc[full_manifest.orig_full_index.isin(good_model_df.index), 'target_f0'].values
good_model_df.loc[: , 'cue_f0'] = full_manifest.loc[full_manifest.orig_full_index.isin(good_model_df.index), 'cue_f0'].values
good_model_df.loc[: , 'distractor_f0'] = full_manifest.loc[full_manifest.orig_full_index.isin(good_model_df.index), 'distractor_f0'].values

good_model_df = good_model_df.reset_index()
good_model_df.rename(columns={'index':'orig_full_index'}, inplace=True)

In [ ]:
sampled_manifest

In [22]:

## Write out good_model_df 
out_name = stim_out_path / 'final_safe_stim_manifest_w_cue_tg_lang_dists_w_transcripts.pdpkl'
# good_model_df.to_pickle(out_name)

In [31]:
good_model_df.head()

,orig_full_index,distractor_client_id,distractor_corpus,distractor_gender,distractor_gender_int,distractor_sr,distractor_src_fn,distractor_word,src_ix,client_id,...,nl_distractor_corpus,zh_distractor_corpus,signal_len_s,word_int,distractor_word_int,target_transcripts,distractor_transcripts,target_f0,cue_f0,distractor_f0
0,0,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,either,992601,1904-cc,...,cv,cv,2,703,210,"[however, the, cost, for, these, new, types, o...","[made, from, either, dna, or]",181.518509,175.263489,189.355301
1,1,matthewdgonzalez,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,seven,992601,1904-cc,...,cv,cv,2,703,610,"[however, the, cost, for, these, new, types, o...","[in, one thousand, nine hundred and forty, sev...",181.518509,175.263489,130.063400
2,2,flyingtoaster,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,were,993029,1904-cc,...,cv,cv,2,623,764,"[so, that, they, can, simply, load, in, the, c...","[of, the, core, rule, books, were, slightly, r...",185.744934,168.147736,196.366379
3,3,warmvoiceover,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,created,993029,1904-cc,...,cv,cv,2,623,166,"[so, that, they, can, simply, load, in, the, c...","[philadelphia, created, in, seventeen]",185.744934,168.147736,107.241776
4,4,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,unknown,1003750,99of9-toby-hudson,...,cv,cv,2,175,747,"[resulted, in, the, death, due, to, a, gas, leak]","[this, was, a, then, unknown, fault]",90.175484,87.435898,222.476532


In [32]:
sampled_manifest.head()

,orig_full_index,distractor_client_id,distractor_corpus,distractor_gender,distractor_gender_int,distractor_sr,distractor_src_fn,distractor_word,src_ix,client_id,...,nl_distractor_corpus,zh_distractor_corpus,signal_len_s,word_int,distractor_word_int,target_transcripts,distractor_transcripts,en_sex_cond,zh_sex_cond,nl_sex_cond
0,626,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,twenty,310886,mangst,...,cv,cv,2,1,736,"[see, particles, above, enceladuss]","[on, october, 29th, nineteen]",different,different,different
1,1276,popularoutcast,swc,female,0,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,noted,181635,the-voice-of-hassocks,...,cv,cv,2,3,458,"[which, according, to, the, constitution]","[the, pact, has, been, noted, yet]",different,different,different
2,928,laura-domineau,swc,female,0,48000,/om/user/imgriff/datasets/spatial_audio_pipeli...,early,126322,popularoutcast,...,cv,cv,2,4,202,"[its, visible, across, interstellar]","[early, years]",same,same,same
3,1049,matthewdgonzalez,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,inside,122796,popularoutcast,...,cv,cv,2,5,338,"[with, the, action]","[line, up, inside, between]",different,different,different
4,1349,enochlau,swc,male,1,44100,/om/user/imgriff/datasets/spatial_audio_pipeli...,worked,940458,vaslittlecrow,...,cv,cv,2,7,784,"[enjoyed, activities, as, well]","[whom, he, had, also, worked, as, a, drafter]",different,different,different


In [35]:
huamn_df_w_f0 = pd.merge(sampled_manifest, good_model_df[['orig_full_index', 'target_f0', 'cue_f0', 'distractor_f0']], left_on='orig_full_index', right_on='orig_full_index')

In [38]:
huamn_df_w_f0['experiment_key_target_word_ix'] = huamn_df_w_f0.index

In [40]:
# huamn_df_w_f0

In [41]:
## Save updated human sampled manifest 
huamn_df_w_f0.to_pickle(stim_out_path / "human_expmt_manifest_w_transcripts_safe_examples.pdpkl" )